# GeoFire-X lot 001: CPU certification
Existing private dataset only. Sequential 1 MiB SHA256/CRC32; 57 train events; no 2021; real rasterio and repository PyTorch DataLoader. No upload, GPU, network, or training.


In [ ]:
import os
os.environ.update(CUDA_VISIBLE_DEVICES='', OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
import base64, hashlib, io, json, sys, zipfile, zlib
from pathlib import Path
work=Path('/kaggle/working/lot001-certification')
work.mkdir(exist_ok=False)
archive=zipfile.ZipFile(io.BytesIO(base64.b64decode('...')))
source_manifest={'geofirex/__init__.py': '41362a5ea4931c546433d4c48290c5707bf02dbbea1c3aafa9f43d1225e47003', 'geofirex/runner_data.py': 'c0acf84bc775c9d8fe06f62080549af74d02a7815a73ef2adc8ccec6371c6f7f', 'geofirex/benchmark.py': '5028ee575056e256fa925d4a20e4a31a4a226243bf3dc8587b9d1a58defff754', 'geofirex/corpus.py': '1677024d71e437985e370b551684ca72a415f1c216b0890864a40f23771bbe52', 'geofirex/remote_archive.py': 'e4c0db84efb0b9460fe0397dd9a809cd6202bcb69af8c9ec9f718c5151940650', 'geofirex/unet_data.py': '78b38ddf3590acb8dc57ff9d5b6071f68b70ad72b4bbd32aee8658b65ca09d1b', 'geofirex/windows.py': '304a8eeca69004bf3dfcd0fb3eb2c3be3a2765b9ac6473673d15109cb4a0a785', 'geofirex/real_constants.py': 'e6d71a56ba0add9aded9fbe01748355f88924aaf33502d4b3b49d12cf55ef1f9'}
for name, sha in source_manifest.items():
    data=archive.read(name)
    if hashlib.sha256(data).hexdigest()!=sha: raise ValueError('Repository source hash mismatch')
    p=work/'source'/name
    p.parent.mkdir(parents=True,exist_ok=True)
    p.write_bytes(data)
sys.path.insert(0,str(work/'source'))
payload=json.loads(zlib.decompress(base64.b64decode('...')))
namespace={'__name__':'lot001_cpu_certification'}
exec(compile('"""Embedded CPU certification runtime. Inputs supplied by the notebook builder."""\nimport datetime as dt\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport sys\nimport threading\nimport time\nimport zlib\n\n\ndef run(reference, observations, normalization, split, out):\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=False)\n    started = time.monotonic()\n    result = dict(status=\'RUNNING\', files_certified=0, bytes_certified=0,\n                  dataset=reference[\'handle\'], identity=reference[\'identity\'],\n                  anomalies=[], gpu_used=False, training=False)\n    (out/\'progress.json\').write_text(json.dumps(result, indent=2))\n    import psutil\n    proc = psutil.Process()\n    metrics = dict(peak_rss_bytes=0, minimum_available_bytes=psutil.virtual_memory().available)\n    stopped = threading.Event()\n    deadline = started + 1800\n\n    def guard():\n        rss, available = proc.memory_info().rss, psutil.virtual_memory().available\n        metrics[\'peak_rss_bytes\'] = max(metrics[\'peak_rss_bytes\'], rss)\n        metrics[\'minimum_available_bytes\'] = min(metrics[\'minimum_available_bytes\'], available)\n        if time.monotonic() > deadline:\n            raise TimeoutError(\'Certification exceeded 1800 seconds\')\n        if rss > 3*1024**3 or available < 2*1024**3:\n            raise MemoryError(\'Certification resource limit reached\')\n\n    def watch():\n        while not stopped.wait(1):\n            try:\n                guard()\n            except Exception as e:\n                (out/\'resource-stop.json\').write_text(json.dumps(dict(status=\'RESOURCE_STOP\', error=str(e), **metrics)))\n                os._exit(75)\n\n    watcher = threading.Thread(target=watch, daemon=True)\n    watcher.start()\n    try:\n        guard()\n        if os.environ.get(\'KAGGLE_ACCELERATOR_TYPE\', \'\').lower() not in (\'\', \'none\', \'cpu\'):\n            raise RuntimeError(\'Accelerated session refused\')\n        if list(Path(\'/dev\').glob(\'nvidia[0-9]*\')):\n            raise RuntimeError(\'GPU device present; CPU session required\')\n        os.environ.update(CUDA_VISIBLE_DEVICES=\'\', OMP_NUM_THREADS=\'2\', MKL_NUM_THREADS=\'2\',\n                          OPENBLAS_NUM_THREADS=\'2\', GDAL_NUM_THREADS=\'1\')\n        from geofirex.runner_data import digest\n        if digest({k:v for k,v in reference.items() if k != \'identity\'}) != reference[\'identity\']:\n            raise ValueError(\'Pinned local manifest identity invalid\')\n        inventories = list(Path(\'/kaggle/input\').rglob(\'lot-inventory.json\'))\n        if len(inventories) != 1:\n            raise ValueError(\'Exactly one attached lot inventory required\')\n        inventory_path = inventories[0]\n        published = json.loads(inventory_path.read_text())\n        if published != reference:\n            raise ValueError(\'Attached inventory differs from local canonical manifest\')\n        root = inventory_path.parent/\'rasters\'\n        expected = {f[\'path\']:f for f in reference[\'files\']}\n        if len(expected) != 1269 or len(reference[\'files\']) != 1269 or reference[\'observations\'] != 1269:\n            raise ValueError(\'Expected exactly 1269 distinct manifest files\')\n        if reference[\'role\'] != \'train\' or len(reference[\'events\']) != 57 or len(set(reference[\'events\'])) != 57:\n            raise ValueError(\'Expected exactly 57 distinct train events\')\n        event_set = {str(PurePosixPath(p).parent) for p in expected}\n        if event_set != set(reference[\'events\']) or any(split.get(e) != \'train\' for e in event_set):\n            raise ValueError(\'Event set or annual train assignment mismatch\')\n        if sum(f[\'bytes\'] for f in expected.values()) != reference[\'bytes\']:\n            raise ValueError(\'Manifest byte total mismatch\')\n        for name in expected:\n            p = PurePosixPath(name)\n            if p.is_absolute() or \'..\' in p.parts or \':\' in name or \'\\\\\' in name or p.parts[0] not in (\'2018\',\'2019\'):\n                raise ValueError(\'Unsafe or non-train canonical raster path\')\n            if dt.date.fromisoformat(p.stem).year != int(p.parts[0]):\n                raise ValueError(\'Observation date/year mismatch\')\n        all_attached = [p for p in Path(\'/kaggle/input\').rglob(\'*\') if p.is_file()]\n        if any(\'2021\' in p.parts or (p.suffix.lower() in (\'.tif\',\'.tiff\') and p.stem.startswith(\'2021\')) for p in all_attached):\n            raise ValueError(\'Held-out 2021 data attached\')\n        actual = {p.relative_to(root).as_posix() for p in root.rglob(\'*\') if p.is_file()}\n        if actual != set(expected) or len(all_attached) != 1270:\n            raise ValueError(f\'Attached file inventory differs: missing={len(set(expected)-actual)}, extra={len(actual-set(expected))}, total={len(all_attached)}\')\n        result[\'inventory\'] = dict(manifest_files=len(published[\'files\']), actual_geotiffs=len(actual),\n            total_attached_files=len(all_attached), events=57, train_events=57,\n            years=sorted({e.split(\'/\')[0] for e in event_set}), heldout_2021_files=0,\n            preview_reported_elements=617,\n            preview_is_complete_manifest=False if len(published[\'files\']) != 617 else True,\n            json_top_level_keys=len(published), manifest_matches_local=True,\n            inventory_sha256=hashlib.sha256(inventory_path.read_bytes()).hexdigest())\n        with (out/\'verified.jsonl\').open(\'x\', encoding=\'utf-8\') as log:\n            for i, entry in enumerate(reference[\'files\'], 1):\n                guard()\n                path = root/entry[\'path\']\n                if path.is_symlink() or not path.resolve().is_relative_to(root.resolve()) or path.stat().st_size != entry[\'bytes\']:\n                    raise ValueError(\'Path or size mismatch: \'+entry[\'path\'])\n                sha, crc, n = hashlib.sha256(), 0, 0\n                with path.open(\'rb\') as source:\n                    for block in iter(lambda: source.read(1024**2), b\'\'):\n                        guard()\n                        sha.update(block)\n                        crc = zlib.crc32(block, crc)\n                        n += len(block)\n                row = dict(path=entry[\'path\'], bytes=n, sha256=sha.hexdigest(), crc32=f\'{crc & 0xffffffff:08x}\')\n                if row != entry:\n                    raise ValueError(\'SHA256/CRC32/size mismatch: \'+entry[\'path\'])\n                log.write(json.dumps(row)+\'\\n\')\n                log.flush()\n                result.update(files_certified=i, bytes_certified=result[\'bytes_certified\']+n)\n                if i % 50 == 0 or i == 1269:\n                    (out/\'progress.json\').write_text(json.dumps(dict(**result, resources=metrics), indent=2))\n                    print(f\'Hash certification {i}/1269; {result["bytes_certified"]} bytes\', flush=True)\n\n        import numpy as np\n        import rasterio\n        import torch\n        from rasterio.windows import Window\n        from torch.utils.data import DataLoader\n        from geofirex.unet_data import Normalizer, validate_layout\n        from geofirex.benchmark import aligned_windows\n        from geofirex.runner_data import FoldData, TemporalStore, TrainTiles\n        torch.set_num_threads(2)\n        torch.set_num_interop_threads(1)\n        # Exercise the actual repository resolver on all 1269 portable tokens.\n        resolver = FoldData.__new__(FoldData)\n        resolver.config = {\'corpus_scope\':\'train_validation\'}\n        resolver.root, resolver.roots = root.resolve(), [root.resolve()]\n        for name in expected:\n            if resolver.resolve_path(name) != (root/name).resolve():\n                raise ValueError(\'Repository DataLoader path resolution differs\')\n        result[\'path_compatibility\'] = dict(resolver=\'geofirex.runner_data.FoldData.resolve_path\',\n            resolved_files=1269, data_root=str(root), data_roots=[str(root)], windows_drive_dependency=False)\n        samples = []\n        indices = (0, len(reference[\'files\'])//4, len(reference[\'files\'])//2, 3*len(reference[\'files\'])//4, -1)\n        with rasterio.Env(GDAL_CACHEMAX=64*1024**2, GDAL_NUM_THREADS=\'1\'):\n            for i in indices:\n                guard()\n                name = reference[\'files\'][i][\'path\']\n                with rasterio.open(root/name) as ds:\n                    validate_layout(ds)\n                    if ds.crs is None:\n                        raise ValueError(\'GeoTIFF CRS missing\')\n                    a = ds.read(window=Window(0,0,min(64,ds.width),min(64,ds.height)))\n                    if a.shape[0] != 23 or np.isinf(a).any():\n                        raise ValueError(\'Raster sample layout/infinite values invalid\')\n                    samples.append(dict(path=name, bands=ds.count, width=ds.width, height=ds.height,\n                        crs=str(ds.crs), transform=list(ds.transform), tile_shape=list(a.shape),\n                        nan_values=int(np.isnan(a).sum()), driver=ds.driver))\n            result[\'rasterio_reads\'] = samples\n            if {r[\'relative_path\'] for r in observations} != set(expected):\n                raise ValueError(\'Local observation QC table differs from lot\')\n            relocated = [{**r,\'relative_path\':str(resolver.resolve_path(r[\'relative_path\']))} for r in observations]\n            groups, rejected = aligned_windows(relocated, split, \'train\', contexts=(1,3,5))\n            if any(not groups[t] for t in (1,3,5)):\n                raise ValueError(\'No valid aligned temporal window\')\n            normalizer = Normalizer(**normalization[\'normalization\'])\n            loader_reads = []\n            for context in (1,3,5):\n                guard()\n                w = groups[context][0]\n                maxw = groups[5][0]\n                geometry = None\n                for path in (*maxw.context_paths, maxw.target_path):\n                    with rasterio.open(path) as ds:\n                        validate_layout(ds)\n                        current = (ds.shape, str(ds.crs), tuple(ds.transform))\n                        if geometry is not None and current != geometry:\n                            raise ValueError(\'Temporal raster geometry mismatch\')\n                        geometry = current\n                store = TemporalStore([w], normalizer, domain_windows=[maxw], cache_bytes=0)\n                dataset = TrainTiles(store, size=64, samples=1, seed=42)\n                x,y,valid = next(iter(DataLoader(dataset,batch_size=1,num_workers=0,pin_memory=False)))\n                if list(x.shape) != [1,context,len(normalizer.channel_names),64,64] or list(y.shape) != [1,1,64,64]:\n                    raise ValueError(\'Actual DataLoader tensor shape mismatch\')\n                if any(t.device.type != \'cpu\' or t.dtype != torch.float32 or not torch.isfinite(t).all() for t in (x,y,valid)):\n                    raise ValueError(\'Actual DataLoader dtype/device/finite check failed\')\n                if not set(torch.unique(y).tolist()) <= {0.,1.} or not set(torch.unique(valid).tolist()) <= {0.,1.}:\n                    raise ValueError(\'Target or validity is not binary\')\n                loader_reads.append(dict(context_days=context, event=w.event_id, context_dates=list(w.context_dates),\n                    target_date=w.target_date, X=list(x.shape), Y=list(y.shape), validity=list(valid.shape),\n                    dtype=str(x.dtype), device=str(x.device), workers=0, cache_bytes=0, finite=True))\n            # Raw 23-band contract, spatially bounded, before feature normalization.\n            w = groups[3][0]\n            raw_days = []\n            for path in w.context_paths:\n                with rasterio.open(path) as ds:\n                    a = ds.read(window=Window(0,0,min(64,ds.width),min(64,ds.height))).astype(np.float32)\n                    a[22] = (~np.isnan(a[22])).astype(np.float32)\n                    a[np.isnan(a)] = 0.\n                    raw_days.append(a)\n            raw = torch.from_numpy(np.stack(raw_days)).unsqueeze(0)\n            if raw.shape[:3] != (1,3,23) or not torch.isfinite(raw).all():\n                raise ValueError(\'Raw temporal contract failed\')\n            result[\'raw_temporal_window\'] = dict(X=list(raw.shape), device=str(raw.device), dtype=str(raw.dtype))\n        guard()\n        result.update(status=\'LOT001_REMOTE_CPU_CERTIFIED\', events=57, role=\'train\',\n            loader_reads=loader_reads, aligned_windows={str(k):len(v) for k,v in groups.items()},\n            window_rejections=rejected, versions=dict(python=sys.version, rasterio=rasterio.__version__,\n                torch=torch.__version__, numpy=np.__version__, psutil=psutil.__version__),\n            resources=dict(**metrics, elapsed_seconds=time.monotonic()-started,\n                deadline_seconds=1800, rss_limit_bytes=3*1024**3, min_available_bytes=2*1024**3,\n                block_bytes=1024**2, gdal_cache_bytes=64*1024**2),\n            scope=\'Only existing attached lot 001; no other lots or training certified\')\n        (out/\'certificate.json\').write_text(json.dumps(result, indent=2))\n        print(json.dumps(result, indent=2), flush=True)\n        return result\n    except Exception as e:\n        result.update(status=\'FAILED_STOP\', error_type=type(e).__name__, error=str(e), resources=metrics)\n        result[\'anomalies\'].append(str(e))\n        (out/\'failure.json\').write_text(json.dumps(result, indent=2))\n        raise\n    finally:\n        stopped.set()\n        watcher.join(timeout=2)\n','certify_kaggle_lot001_runtime.py','exec'),namespace)
certificate=namespace['run'](**payload,out=work/'results')
